In [13]:
# -*- coding: utf-8 -*-
"""
股票、指数、宏观数据获取脚本（最终修正版）
功能：下载股票、指数、宏观数据，并保存到指定目录
"""

import os
import pandas as pd
import numpy as np
import baostock as bs
import akshare as ak
from datetime import datetime
import time
import random
import re
import warnings
warnings.filterwarnings('ignore')

# 创建必要的文件夹
def create_directories():
    """创建所有需要的文件夹"""
    folders = ['data/stock', 'data/index', 'data/macro', 'data/finance', 'data/combined']
    for folder in folders:
        os.makedirs(folder, exist_ok=True)
    print("所有文件夹创建完成")

# 日志记录函数
def write_log(message):
    """写入日志文件"""
    with open('download_log.txt', 'a', encoding='utf-8') as f:
        f.write(message + '\n')
    print(message)

# 股票列表
stocks = [
    {'code': '600036', 'name': '招商银行'},
    {'code': '600104', 'name': '上汽集团'},
    {'code': '000002', 'name': '万科A'},
    {'code': '600048', 'name': '保利地产'},
    {'code': '600519', 'name': '贵州茅台'},
    {'code': '000858', 'name': '五粮液'},
    {'code': '601857', 'name': '中国石油'},
    {'code': '601898', 'name': '中煤能源'},
    {'code': '600050', 'name': '中国联通'},
    {'code': '600233', 'name': '圆通速递'}
]

# 指数列表
indices = [
    {'code': 'sh.000300', 'name': '000300'},  # 沪深300
    {'code': 'sz.399006', 'name': '399006'}   # 创业板指
]

def download_stock_data(stock_code, start_date='2020-01-01', end_date='2026-04-01'):
    """
    下载单只股票的后复权日度行情数据
    """
    max_retries = 3
    for attempt in range(max_retries):
        try:
            # 登录baostock
            lg = bs.login()
            if lg.error_code != '0':
                if attempt < max_retries - 1:
                    time.sleep(2)
                    continue
                return None, f"Error: Login failed - {lg.error_msg}"
            
            # 获取股票代码（baostock需要sh/sz前缀）
            if stock_code.startswith('6'):
                bs_code = f'sh.{stock_code}'
            else:
                bs_code = f'sz.{stock_code}'
            
            # 获取后复权数据
            rs = bs.query_history_k_data_plus(
                bs_code,
                "date,open,high,low,close,volume,amount",
                start_date=start_date,
                end_date=end_date,
                frequency="d",
                adjustflag="1"  # 1表示后复权
            )
            
            if rs.error_code != '0':
                bs.logout()
                if attempt < max_retries - 1:
                    time.sleep(2)
                    continue
                return None, f"Error: {rs.error_msg}"
            
            # 转换为DataFrame
            data_list = []
            while (rs.error_code == '0') & rs.next():
                data_list.append(rs.get_row_data())
            
            if not data_list:
                bs.logout()
                return None, "Error: No data returned"
            
            df = pd.DataFrame(data_list, columns=rs.fields)
            
            # 转换数据类型
            for col in ['open', 'high', 'low', 'close', 'volume', 'amount']:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
            
            df['date'] = pd.to_datetime(df['date'])
            
            bs.logout()
            return df, "SUCCESS"
            
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2)
                continue
            return None, f"Error: {str(e)}"
    
    return None, "Error: Max retries exceeded"

def download_index_data(index_code, start_date='2020-01-01', end_date='2026-04-01'):
    """
    下载指数日度数据
    """
    max_retries = 3
    for attempt in range(max_retries):
        try:
            # 登录baostock
            lg = bs.login()
            
            # 获取指数数据
            rs = bs.query_history_k_data_plus(
                index_code,
                "date,open,high,low,close,volume,amount",
                start_date=start_date,
                end_date=end_date,
                frequency="d"
            )
            
            if rs.error_code != '0':
                bs.logout()
                if attempt < max_retries - 1:
                    time.sleep(2)
                    continue
                return None, f"Error: {rs.error_msg}"
            
            # 转换为DataFrame
            data_list = []
            while (rs.error_code == '0') & rs.next():
                data_list.append(rs.get_row_data())
            
            if not data_list:
                bs.logout()
                return None, "Error: No data returned"
            
            df = pd.DataFrame(data_list, columns=rs.fields)
            
            # 转换数据类型
            for col in ['open', 'high', 'low', 'close', 'volume', 'amount']:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
            
            df['date'] = pd.to_datetime(df['date'])
            
            bs.logout()
            return df, "SUCCESS"
            
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2)
                continue
            return None, f"Error: {str(e)}"
    
    return None, "Error: Max retries exceeded"

def download_macro_data():
    """
    下载CPI和M2月度数据
    CPI使用 ak.macro_china_cpi()
    M2使用 ak.macro_china_money_supply()
    """
    macro_data = {'cpi': None, 'm2': None}
    
    # 下载CPI月度数据
    try:
        print("正在下载CPI月度数据...")
        cpi_df = ak.macro_china_cpi()
        
        if cpi_df is not None and not cpi_df.empty:
            print(f"CPI数据列名: {cpi_df.columns.tolist()}")
            print(f"CPI数据前几行:\n{cpi_df.head()}")
            
            # 直接使用原始数据，不解析日期
            cpi_result = pd.DataFrame()
            cpi_result['date'] = cpi_df['月份']
            cpi_result['cpi_yoy'] = cpi_df['全国-同比增长']
            
            # 筛选2020-2026年的数据
            cpi_result = cpi_result[cpi_result['date'].str.contains('2020|2021|2022|2023|2024|2025|2026')]
            
            if not cpi_result.empty:
                macro_data['cpi'] = cpi_result
                print(f"CPI数据下载成功，共{len(cpi_result)}条记录")
                print(f"CPI数据预览:\n{cpi_result.head()}")
            else:
                print("CPI数据在筛选后为空")
        else:
            print("CPI数据下载失败：未获取到数据")
            
    except Exception as e:
        print(f"CPI数据下载失败: {str(e)}")
    
    time.sleep(2)
    
    # 下载M2月度数据
    try:
        print("正在下载M2月度数据...")
        m2_df = ak.macro_china_money_supply()
        
        if m2_df is not None and not m2_df.empty:
            print(f"M2数据列名: {m2_df.columns.tolist()}")
            print(f"M2数据前几行:\n{m2_df.head()}")
            
            # 直接使用原始数据，不解析日期
            m2_result = pd.DataFrame()
            m2_result['date'] = m2_df['月份']
            m2_result['m2_yoy'] = m2_df['货币和准货币(M2)-同比增长']
            
            # 筛选2020-2026年的数据
            m2_result = m2_result[m2_result['date'].str.contains('2020|2021|2022|2023|2024|2025|2026')]
            
            if not m2_result.empty:
                macro_data['m2'] = m2_result
                print(f"M2数据下载成功，共{len(m2_result)}条记录")
                print(f"M2数据预览:\n{m2_result.head()}")
            else:
                print("M2数据在筛选后为空")
        else:
            print("M2数据下载失败：未获取到数据")
            
    except Exception as e:
        print(f"M2数据下载失败: {str(e)}")
    
    return macro_data

def download_financial_ratios(stock_codes, start_year=2021, end_year=2025):
    """
    下载ROE和资产负债率数据
    """
    try:
        lg = bs.login()
        if lg.error_code != '0':
            return None, f"Error: Baostock login failed - {lg.error_msg}"
        
        all_data = []
        
        for stock_code in stock_codes:
            print(f"正在获取 {stock_code} 的财务数据...")
            
            # 确定代码前缀
            if stock_code.startswith('6'):
                bs_code = f'sh.{stock_code}'
            else:
                bs_code = f'sz.{stock_code}'
            
            # 获取资产负债率
            try:
                for year in range(start_year, end_year + 1):
                    rs_balance = bs.query_balance_data(bs_code, year, 4)  # 4表示年报
                    
                    if rs_balance.error_code == '0':
                        while rs_balance.next():
                            row = rs_balance.get_row_data()
                            
                            # 获取资产负债率 - 字段名 liabilityToAsset
                            debt_ratio = None
                            for i, field in enumerate(rs_balance.fields):
                                if field == 'liabilityToAsset':
                                    if i < len(row) and row[i] and row[i] != '':
                                        debt_ratio = float(row[i])
                                        break
                            
                            if debt_ratio is not None:
                                all_data.append({
                                    'code': stock_code,
                                    'year': str(year),
                                    'indicator': 'Debt_Ratio',
                                    'value': round(debt_ratio, 4)
                                })
                                print(f"  获取到 {stock_code} {year}年资产负债率: {debt_ratio:.4f}")
                            break
                    else:
                        print(f"  资产负债表查询失败({year}): {rs_balance.error_msg}")
                    
                    time.sleep(0.5)
                    
            except Exception as e:
                print(f"  获取{stock_code}资产负债率失败: {str(e)}")
            
            # 获取ROE
            try:
                for year in range(start_year, end_year + 1):
                    rs_profit = bs.query_profit_data(bs_code, year, 4)  # 4表示年报
                    
                    if rs_profit.error_code == '0':
                        while rs_profit.next():
                            row = rs_profit.get_row_data()
                            
                            # 查找ROE字段
                            roe_value = None
                            for i, field in enumerate(rs_profit.fields):
                                if 'roe' in field.lower() or field == 'profitStatementROE':
                                    if i < len(row) and row[i] and row[i] != '':
                                        roe_value = float(row[i])
                                        break
                            
                            if roe_value is not None:
                                all_data.append({
                                    'code': stock_code,
                                    'year': str(year),
                                    'indicator': 'ROE',
                                    'value': round(roe_value, 4)
                                })
                                print(f"  获取到 {stock_code} {year}年ROE: {roe_value:.2f}%")
                            break
                    else:
                        print(f"  盈利能力查询失败({year}): {rs_profit.error_msg}")
                    
                    time.sleep(0.5)
                    
            except Exception as e:
                print(f"  获取{stock_code} ROE失败: {str(e)}")
            
            # 随机延时
            time.sleep(random.uniform(1, 2))
        
        bs.logout()
        
        if all_data:
            df = pd.DataFrame(all_data)
            df = df.sort_values(['code', 'year', 'indicator'])
            print(f"\n财务数据汇总: 共{len(df)}条记录")
            return df, "SUCCESS"
        else:
            return None, "Error: No financial data retrieved"
            
    except Exception as e:
        if 'bs' in locals():
            bs.logout()
        return None, f"Error: {str(e)}"

def combine_all_data():
    """
    合并所有数据到combined文件夹
    """
    try:
        print("\n--- 开始合并数据 ---")
        all_dfs = []
        
        # 合并股票数据
        stock_files = [f for f in os.listdir('data/stock') if f.startswith('stock_') and f.endswith('.csv')]
        for file in stock_files:
            file_path = os.path.join('data/stock', file)
            if os.path.exists(file_path):
                df = pd.read_csv(file_path)
                df['data_type'] = 'stock'
                df['code'] = file.replace('stock_', '').replace('.csv', '')
                all_dfs.append(df)
                print(f"  已加载股票数据: {file} (行数: {len(df)})")
        
        # 合并指数数据
        index_files = [f for f in os.listdir('data/index') if f.startswith('index_') and f.endswith('.csv')]
        for file in index_files:
            file_path = os.path.join('data/index', file)
            if os.path.exists(file_path):
                df = pd.read_csv(file_path)
                df['data_type'] = 'index'
                df['code'] = file.replace('index_', '').replace('.csv', '')
                all_dfs.append(df)
                print(f"  已加载指数数据: {file} (行数: {len(df)})")
        
        # 合并宏观数据
        macro_files = [f for f in os.listdir('data/macro') if f.startswith('macro_') and f.endswith('.csv')]
        for file in macro_files:
            file_path = os.path.join('data/macro', file)
            if os.path.exists(file_path):
                df = pd.read_csv(file_path)
                df['data_type'] = 'macro'
                indicator = file.replace('macro_', '').replace('.csv', '')
                df['indicator'] = indicator
                all_dfs.append(df)
                print(f"  已加载宏观数据: {file} (行数: {len(df)})")
        
        # 合并财务数据
        finance_file = 'data/finance/finance_ratios.csv'
        if os.path.exists(finance_file):
            df = pd.read_csv(finance_file)
            df['data_type'] = 'finance'
            all_dfs.append(df)
            print(f"  已加载财务数据: finance_ratios.csv (行数: {len(df)})")
        
        # 保存合并后的数据
        if all_dfs:
            combined_df = pd.concat(all_dfs, ignore_index=True, sort=False)
            combined_file = 'data/combined/combined_data.csv'
            combined_df.to_csv(combined_file, index=False, encoding='utf-8-sig')
            print(f"  数据合并完成，总记录数: {len(combined_df)}")
            return combined_df, f"SUCCESS shape={combined_df.shape}"
        else:
            return None, "Error: No data to combine"
            
    except Exception as e:
        return None, f"Error: {str(e)}"

def main():
    """主函数"""
    start_time = datetime.now()
    write_log(f"\n========== 开始数据下载任务 {start_time.strftime('%Y-%m-%d %H:%M:%S')} ==========")
    
    # 创建文件夹
    create_directories()
    
    # 1. 下载股票数据
    write_log("\n--- 开始下载股票数据 ---")
    
    for stock in stocks:
        print(f"\n正在下载 {stock['name']}({stock['code']}) 数据...")
        df, status = download_stock_data(stock['code'])
        
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        if df is not None and status == "SUCCESS":
            filename = f"data/stock/stock_{stock['code']}.csv"
            df.to_csv(filename, index=False, encoding='utf-8-sig')
            log_msg = f"[{timestamp}] SUCCESS  stock_{stock['code']}  shape={df.shape}"
            write_log(log_msg)
        else:
            log_msg = f"[{timestamp}] FAILED   stock_{stock['code']}  {status}"
            write_log(log_msg)
        
        time.sleep(random.uniform(0.5, 1.5))
    
    # 2. 下载指数数据
    write_log("\n--- 开始下载指数数据 ---")
    
    for index in indices:
        print(f"\n正在下载指数 {index['name']} 数据...")
        df, status = download_index_data(index['code'])
        
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        if df is not None and status == "SUCCESS":
            filename = f"data/index/index_{index['name']}.csv"
            df.to_csv(filename, index=False, encoding='utf-8-sig')
            log_msg = f"[{timestamp}] SUCCESS  index_{index['name']}  shape={df.shape}"
            write_log(log_msg)
        else:
            log_msg = f"[{timestamp}] FAILED   index_{index['name']}  {status}"
            write_log(log_msg)
        
        time.sleep(random.uniform(0.5, 1.5))
    
    # 3. 下载宏观数据
    write_log("\n--- 开始下载宏观数据 ---")
    macro_data = download_macro_data()
    
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    if macro_data['cpi'] is not None and not macro_data['cpi'].empty:
        macro_data['cpi'].to_csv('data/macro/macro_cpi.csv', index=False, encoding='utf-8-sig')
        log_msg = f"[{timestamp}] SUCCESS  macro_cpi  shape={macro_data['cpi'].shape}"
        write_log(log_msg)
    else:
        log_msg = f"[{timestamp}] FAILED   macro_cpi  Error: Download failed"
        write_log(log_msg)
    
    if macro_data['m2'] is not None and not macro_data['m2'].empty:
        macro_data['m2'].to_csv('data/macro/macro_m2.csv', index=False, encoding='utf-8-sig')
        log_msg = f"[{timestamp}] SUCCESS  macro_m2  shape={macro_data['m2'].shape}"
        write_log(log_msg)
    else:
        log_msg = f"[{timestamp}] FAILED   macro_m2  Error: Download failed"
        write_log(log_msg)
    
    # 4. 下载财务比率数据
    write_log("\n--- 开始下载财务比率数据 ---")
    stock_codes = [stock['code'] for stock in stocks]
    df_finance, status = download_financial_ratios(stock_codes)
    
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    if df_finance is not None and status == "SUCCESS":
        df_finance.to_csv('data/finance/finance_ratios.csv', index=False, encoding='utf-8-sig')
        log_msg = f"[{timestamp}] SUCCESS  finance_ratios  shape={df_finance.shape}"
        write_log(log_msg)
        print(f"\n财务数据预览:\n{df_finance.head(10)}")
    else:
        log_msg = f"[{timestamp}] FAILED   finance_ratios  {status}"
        write_log(log_msg)
    
    # 5. 合并数据
    combined_df, status = combine_all_data()
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    if "SUCCESS" in status:
        log_msg = f"[{timestamp}] SUCCESS  combined_data  {status}"
        write_log(log_msg)
    else:
        log_msg = f"[{timestamp}] FAILED   combined_data  {status}"
        write_log(log_msg)
    
    end_time = datetime.now()
    write_log(f"\n========== 数据下载任务完成 {end_time.strftime('%Y-%m-%d %H:%M:%S')} 总耗时: {end_time - start_time} ==========")
    
    print("\n所有任务完成！请查看download_log.txt了解详细信息。")

# 运行主函数
if __name__ == "__main__":
    main()


========== 开始数据下载任务 2026-04-03 14:19:47 ==========
所有文件夹创建完成

--- 开始下载股票数据 ---

正在下载 招商银行(600036) 数据...
login success!
logout success!
[2026-04-03 14:19:52] SUCCESS  stock_600036  shape=(1512, 7)

正在下载 上汽集团(600104) 数据...
login success!
logout success!
[2026-04-03 14:19:56] SUCCESS  stock_600104  shape=(1512, 7)

正在下载 万科A(000002) 数据...
login success!
logout success!
[2026-04-03 14:19:59] SUCCESS  stock_000002  shape=(1512, 7)

正在下载 保利地产(600048) 数据...
login success!
logout success!
[2026-04-03 14:20:01] SUCCESS  stock_600048  shape=(1512, 7)

正在下载 贵州茅台(600519) 数据...
login success!
logout success!
[2026-04-03 14:20:04] SUCCESS  stock_600519  shape=(1512, 7)

正在下载 五粮液(000858) 数据...
login success!
logout success!
[2026-04-03 14:20:10] SUCCESS  stock_000858  shape=(1512, 7)

正在下载 中国石油(601857) 数据...
login success!
logout success!
[2026-04-03 14:20:12] SUCCESS  stock_601857  shape=(1512, 7)

正在下载 中煤能源(601898) 数据...
login success!
logout success!
[2026-04-03 14:20:16] SUCCESS  stock_601898  sh

### 提示词：
- 我在用vscode编程,代码保存在.ipynb,请实现以下功能，
在当前文件夹下创建data文件夹.
使用baostock获取 招商银行(600036),上汽集团(600104 ),万科A(000002),保利地产(600048),贵州茅台(600519),五粮液(000858),中国石油(601857),中煤能源(601898),中国联通(600050),圆通速递(600233),这些股票过去 5 年（2020-01-01 至2026-04-01 ）的后复权日度行情，字段须包含：日期、开盘价、收盘价、最高价、最低价、成交量、成交额.在当前目录下data文件夹中创建stock文件夹,每只股票保存在stock文件夹中类似stock_000001.csv命名方式的文件中.
使用baostock下载沪深 300（000300）,创业板指（399006）2020-01-01 至2026-04-01 的日度数据,在当前目录下data文件夹中创建index文件夹,每只股票保存在index文件夹中类似index_000300.csv命名方式的文件中.
用 akshare下载CPI 同比增速,M2 同比增速在2020-01-01 至2026-04-01 的的月度数据,在当前目录下data文件夹中创建macro文件夹,每只股票保存在macro文件夹中类似macro_cpi.csv,macro_M2.csv命名方式的文件中.
使用baostock获取上述 10 只股票最近 5 个年度净资产收益率（ROE）,资产负债率,并将数据整理为长格式（Long format）：每行为一只股票一个年度的观测，字段包含 code, year, indicator, value.在当前目录下data文件夹中创建finance文件夹,数据保存在finance文件夹中finance_ratios.csv命名的文件中.
将每次下载的结果记录到 当前目录下的download_log.txt文件中，格式如下：
[2025-05-01 10:23:45] SUCCESS  stock_000001  shape=(1200, 7)
[2025-05-01 10:23:48] SUCCESS  stock_600519  shape=(1198, 7)
[2025-05-01 10:23:51] FAILED   stock_999999  Error: No data returned
注意不要触发网页反爬机制。
当前目录下data文件夹中创建combined文件夹,将上述股票，指数，宏观数据合并保存在combined_data.csv文件中.